In [4]:
# -*- coding: utf-8 -*-
"""
Validation Step 3 — Conclusion-Robustness Checks
ICST/MSR emulator testing study (current methodology-aligned)

What this script does
---------------------
1) Controller-regime robustness
   Compares the study's main controlled subset against adjacent regimes.

2) Signature-based within-shape robustness
   Re-runs the main style summaries inside selected high-support workflow-shape signatures.

Outputs
-------
Creates a folder with:
- step3_regime_summary.csv
- step3_regime_ordering_vs_base.csv
- step3_signature_candidates.csv
- step3_selected_signature_summary.csv
- step3_signature_conclusion_stability.csv
- step3_notes.txt
"""

from pathlib import Path
import numpy as np
import pandas as pd

# ============================================================
# CONFIG
# ============================================================
BASE_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext")

IN_MAIN = BASE_DIR / "MainDataset.csv"

OUT_DIR = BASE_DIR / r"0.2-Validation\Step 3 - Robustness_Check"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Signature selection thresholds
MIN_SIGNATURE_TOTAL_N = 80
MIN_SIGNATURE_STYLE_N = 15
MIN_SIGNATURE_USABLE_STYLES = 2
TOP_K_SIGNATURES = 5

# Style order reference
STYLE_ORDER_ALL = ["Community", "Third-Party", "GMD", "Custom"]

# Primary timing measures from the current methodology
LAYER1_MEASURES = {
    "run_duration": "study_run_duration_seconds",
    "l1_time_to_instr_env": "study_layer1_time_to_instrumentation_envelope_seconds",
    "l1_instr_job_env": "study_layer1_instrumentation_job_envelope_seconds",
    "l1_post_instr_tail": "study_layer1_post_instrumentation_tail_seconds",
}

LAYER2_MEASURES = {
    "l2_pre_invocation": "study_pre_invocation_selected_stage3_seconds",
    "l2_execution_window": "study_invocation_execution_window_selected_stage3_seconds",
    "l2_post_invocation": "study_post_invocation_selected_stage3_seconds",
}

GROUPING_SUM_FIELDS = {
    "setup_sum": "study_setup_sum_seconds",
    "provision_sum": "study_provision_sum_seconds",
    "test_sum": "study_test_sum_seconds",
    "artifact_report_sum": "study_artifact_report_sum_seconds",
    "cleanup_teardown_sum": "study_cleanup_teardown_sum_seconds",
    "other_sum": "study_other_sum_seconds",
    "execution_related_sum": "study_execution_related_sum_seconds",
    "non_execution_overhead_sum": "study_non_execution_overhead_sum_seconds",
    "pre_test_overhead_sum": "study_pre_test_overhead_sum_seconds",
    "active_test_sum": "study_active_test_sum_seconds",
    "post_test_overhead_sum": "study_post_test_overhead_sum_seconds",
}

SIGNATURE_FIELDS = [
    "study_signature_hash",
    "study_runner_os_bucket",
    "study_job_count_total_bucket",
    "study_step_count_exec_bucket",
]

# ============================================================
# HELPERS
# ============================================================
def norm_bool(series: pd.Series) -> pd.Series:
    s = series.copy()
    if pd.api.types.is_bool_dtype(s):
        return s.astype("boolean")
    s = s.astype(str).str.strip().str.lower()
    mapping = {
        "true": True, "false": False,
        "1": True, "0": False,
        "yes": True, "no": False,
        "y": True, "n": False,
    }
    out = s.map(mapping)
    return out.astype("boolean")


def to_num(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def pct95(series: pd.Series) -> float:
    s = series.dropna()
    if s.empty:
        return np.nan
    return float(np.percentile(s, 95))


def iqr(series: pd.Series) -> float:
    s = series.dropna()
    if s.empty:
        return np.nan
    return float(np.percentile(s, 75) - np.percentile(s, 25))


def safe_ratio(a, b):
    if pd.isna(a) or pd.isna(b) or b == 0:
        return np.nan
    return float(a / b)


def median_style_order(summary_df: pd.DataFrame, regime: str, measure_key: str, min_n: int = 10) -> list:
    part = summary_df[
        (summary_df["regime"] == regime) &
        (summary_df["measure"] == measure_key) &
        (summary_df["n_non_missing"] >= min_n)
    ].copy()

    if part.empty:
        return []

    part["style_rank"] = part["style"].map({s: i for i, s in enumerate(STYLE_ORDER_ALL)})
    part = part.sort_values(["median", "style_rank"], na_position="last")
    return part["style"].tolist()


def compare_orderings(reference_order: list, other_order: list) -> str:
    if len(other_order) < 2:
        return "not_testable"
    if other_order == reference_order:
        return "preserved"
    if len(reference_order) >= 1 and len(other_order) >= 1 and reference_order[0] == other_order[0]:
        return "partially_preserved"
    if len(reference_order) >= 2 and len(other_order) >= 2 and set(reference_order[:2]) == set(other_order[:2]):
        return "partially_preserved"
    return "changed"


def dominant_bucket_from_sums(group: pd.DataFrame, columns_map: dict) -> str:
    meds = {}
    for k, c in columns_map.items():
        if c in group.columns:
            meds[k] = group[c].median(skipna=True)
    if not meds:
        return ""
    meds = {k: v for k, v in meds.items() if not pd.isna(v)}
    if not meds:
        return ""
    return max(meds.items(), key=lambda x: x[1])[0]


def add_measure_summary(rows, df_part, regime_name: str, layer_name: str, measures: dict):
    for measure_key, col in measures.items():
        if col not in df_part.columns:
            continue

        for style, g in df_part.groupby("style", dropna=False):
            s = to_num(g[col])
            n_total = len(g)
            n_non_missing = int(s.notna().sum())
            med = float(s.median()) if n_non_missing else np.nan
            q1 = float(s.quantile(0.25)) if n_non_missing else np.nan
            q3 = float(s.quantile(0.75)) if n_non_missing else np.nan
            p95 = pct95(s) if n_non_missing else np.nan
            iqr_val = iqr(s) if n_non_missing else np.nan

            rows.append({
                "regime": regime_name,
                "layer": layer_name,
                "measure": measure_key,
                "column_name": col,
                "style": style,
                "n_total_records": n_total,
                "n_non_missing": n_non_missing,
                "coverage_rate": safe_ratio(n_non_missing, n_total),
                "median": med,
                "q1": q1,
                "q3": q3,
                "iqr": iqr_val,
                "p95": p95,
                "iqr_over_median": safe_ratio(iqr_val, med),
                "p95_over_median": safe_ratio(p95, med),
            })

# ============================================================
# LOAD + NORMALIZE
# ============================================================
if not IN_MAIN.exists():
    raise FileNotFoundError(f"MainDataset.csv not found at: {IN_MAIN}")

df = pd.read_csv(IN_MAIN, low_memory=False)

for c in [
    "Base", "Robust", "controller_attempt_eq_1", "controller_run_verdict_complete",
    "controller_style_in_scope", "controller_instru_job_count_gt0"
]:
    if c in df.columns:
        df[c] = norm_bool(df[c])

if "run_attempt" in df.columns:
    df["run_attempt"] = pd.to_numeric(df["run_attempt"], errors="coerce")

for col in list(LAYER1_MEASURES.values()) + list(LAYER2_MEASURES.values()) + list(GROUPING_SUM_FIELDS.values()):
    if col in df.columns:
        df[col] = to_num(df[col])

if "study_layer2_measurement_mode" in df.columns:
    df["layer2_observable"] = df["study_layer2_measurement_mode"].astype(str).str.strip().eq("measured_step_based")
else:
    first_l2 = next(iter(LAYER2_MEASURES.values()))
    df["layer2_observable"] = df[first_l2].notna() if first_l2 in df.columns else False

# ============================================================
# DEFINE REGIMES
# ============================================================
all_run_per_style_mask = pd.Series(True, index=df.index)
if "controller_style_in_scope" in df.columns:
    all_run_per_style_mask &= df["controller_style_in_scope"].fillna(False)
if "controller_instru_job_count_gt0" in df.columns:
    all_run_per_style_mask &= df["controller_instru_job_count_gt0"].fillna(False)

verdict_complete_mask = all_run_per_style_mask.copy()
if "controller_run_verdict_complete" in df.columns:
    verdict_complete_mask &= df["controller_run_verdict_complete"].fillna(False)

controlled_subset_mask = df["Base"].fillna(False) if "Base" in df.columns else (
    all_run_per_style_mask &
    df["controller_run_verdict_complete"].fillna(False) &
    df["controller_attempt_eq_1"].fillna(False)
)

rerun_verdict_complete_mask = verdict_complete_mask & (df["run_attempt"].fillna(1) > 1)

regimes = {
    "all_run_per_style": all_run_per_style_mask,
    "verdict_complete": verdict_complete_mask,
    "controlled_subset": controlled_subset_mask,
    "rerun_verdict_complete": rerun_verdict_complete_mask,
}

REFERENCE_REGIME = "controlled_subset"

# ============================================================
# CHECK A — CONTROLLER-REGIME ROBUSTNESS
# ============================================================
summary_rows = []

for regime_name, mask in regimes.items():
    part = df.loc[mask].copy()
    if part.empty:
        continue

    add_measure_summary(summary_rows, part, regime_name, "layer1", LAYER1_MEASURES)
    add_measure_summary(
        summary_rows,
        part.loc[part["layer2_observable"]].copy(),
        regime_name,
        "layer2",
        LAYER2_MEASURES
    )

regime_summary = pd.DataFrame(summary_rows)
regime_summary.to_csv(OUT_DIR / "step3_regime_summary.csv", index=False)

ordering_rows = []
for layer_name, measures in [("layer1", LAYER1_MEASURES), ("layer2", LAYER2_MEASURES)]:
    for measure_key in measures.keys():
        reference_order = median_style_order(regime_summary, REFERENCE_REGIME, measure_key, min_n=10)
        for regime_name in regimes.keys():
            other_order = median_style_order(regime_summary, regime_name, measure_key, min_n=10)
            ordering_rows.append({
                "layer": layer_name,
                "measure": measure_key,
                "reference_regime": REFERENCE_REGIME,
                "target_regime": regime_name,
                "reference_order": " > ".join(reference_order) if reference_order else "",
                "target_order": " > ".join(other_order) if other_order else "",
                "ordering_status_vs_reference": compare_orderings(reference_order, other_order),
            })

regime_ordering = pd.DataFrame(ordering_rows)
regime_ordering.to_csv(OUT_DIR / "step3_regime_ordering_vs_base.csv", index=False)

# ============================================================
# CHECK B — SIGNATURE-BASED WITHIN-SHAPE ROBUSTNESS
# ============================================================
if "Robust" in df.columns and df["Robust"].notna().any():
    signature_pool_mask = df["Robust"].fillna(False)
    signature_pool_name = "robust"
else:
    signature_pool_mask = controlled_subset_mask
    signature_pool_name = REFERENCE_REGIME

sig_pool = df.loc[signature_pool_mask].copy()

if "study_signature_hash" not in sig_pool.columns:
    raise ValueError("Missing 'study_signature_hash' column required for signature-based Step 3.")

sig_counts = (
    sig_pool.groupby(["study_signature_hash", "style"], dropna=False)
    .size()
    .rename("n")
    .reset_index()
)

sig_wide = sig_counts.pivot_table(index="study_signature_hash", columns="style", values="n", fill_value=0)
sig_wide = sig_wide.reset_index()

for s in STYLE_ORDER_ALL:
    if s not in sig_wide.columns:
        sig_wide[s] = 0

sig_wide["total_n"] = sig_wide[STYLE_ORDER_ALL].sum(axis=1)
sig_wide["usable_style_count"] = (sig_wide[STYLE_ORDER_ALL] >= MIN_SIGNATURE_STYLE_N).sum(axis=1)

rep_cols = ["study_signature_hash"] + [c for c in [
    "study_runner_os_bucket", "study_job_count_total_bucket", "study_step_count_exec_bucket",
    "study_sig_basis_base", "study_signature_inputs"
] if c in sig_pool.columns]

sig_meta = (
    sig_pool[rep_cols]
    .drop_duplicates(subset=["study_signature_hash"])
    .copy()
)

sig_candidates = sig_wide.merge(sig_meta, on="study_signature_hash", how="left")
sig_candidates["selected_pool"] = signature_pool_name
sig_candidates["eligible"] = (
    (sig_candidates["total_n"] >= MIN_SIGNATURE_TOTAL_N) &
    (sig_candidates["usable_style_count"] >= MIN_SIGNATURE_USABLE_STYLES)
)

sig_candidates = sig_candidates.sort_values(
    ["eligible", "total_n", "usable_style_count"],
    ascending=[False, False, False]
).reset_index(drop=True)

sig_candidates.to_csv(OUT_DIR / "step3_signature_candidates.csv", index=False)

selected_signatures = sig_candidates.loc[
    sig_candidates["eligible"]
].head(TOP_K_SIGNATURES)["study_signature_hash"].tolist()

selected_rows = []
stability_rows = []

for sig in selected_signatures:
    sig_part = sig_pool.loc[sig_pool["study_signature_hash"] == sig].copy()

    for style, g in sig_part.groupby("style", dropna=False):
        for measure_key, col in LAYER1_MEASURES.items():
            s = g[col] if col in g.columns else pd.Series(dtype=float)
            s = to_num(s)
            n_non_missing = int(s.notna().sum())
            med = float(s.median()) if n_non_missing else np.nan
            iqr_val = iqr(s) if n_non_missing else np.nan
            p95_val = pct95(s) if n_non_missing else np.nan

            selected_rows.append({
                "study_signature_hash": sig,
                "layer": "layer1",
                "measure": measure_key,
                "style": style,
                "n_total_records": len(g),
                "n_non_missing": n_non_missing,
                "coverage_rate": safe_ratio(n_non_missing, len(g)),
                "median": med,
                "iqr": iqr_val,
                "p95": p95_val,
                "iqr_over_median": safe_ratio(iqr_val, med),
                "p95_over_median": safe_ratio(p95_val, med),
            })

        g2 = g.loc[g["layer2_observable"]].copy()
        for measure_key, col in LAYER2_MEASURES.items():
            s = g2[col] if col in g2.columns else pd.Series(dtype=float)
            s = to_num(s)
            n_non_missing = int(s.notna().sum())
            med = float(s.median()) if n_non_missing else np.nan
            iqr_val = iqr(s) if n_non_missing else np.nan
            p95_val = pct95(s) if n_non_missing else np.nan

            selected_rows.append({
                "study_signature_hash": sig,
                "layer": "layer2",
                "measure": measure_key,
                "style": style,
                "n_total_records": len(g2),
                "n_non_missing": n_non_missing,
                "coverage_rate": safe_ratio(n_non_missing, len(g2)),
                "median": med,
                "iqr": iqr_val,
                "p95": p95_val,
                "iqr_over_median": safe_ratio(iqr_val, med),
                "p95_over_median": safe_ratio(p95_val, med),
            })

    temp_sig_summary = pd.DataFrame([r for r in selected_rows if r["study_signature_hash"] == sig])
    sig_meta_one = sig_candidates.loc[sig_candidates["study_signature_hash"] == sig].iloc[0].to_dict()

    for layer_name, measures in [("layer1", LAYER1_MEASURES), ("layer2", LAYER2_MEASURES)]:
        for measure_key in measures.keys():
            reference_order = median_style_order(regime_summary, REFERENCE_REGIME, measure_key, min_n=10)

            part = temp_sig_summary[
                (temp_sig_summary["layer"] == layer_name) &
                (temp_sig_summary["measure"] == measure_key) &
                (temp_sig_summary["n_non_missing"] >= MIN_SIGNATURE_STYLE_N)
            ].copy()

            if not part.empty:
                part["style_rank"] = part["style"].map({s: i for i, s in enumerate(STYLE_ORDER_ALL)})
                part = part.sort_values(["median", "style_rank"])
                sig_order = part["style"].tolist()
            else:
                sig_order = []

            dom_grouping = dominant_bucket_from_sums(sig_part, GROUPING_SUM_FIELDS)

            stability_rows.append({
                "study_signature_hash": sig,
                "selected_pool": signature_pool_name,
                "layer": layer_name,
                "measure": measure_key,
                "reference_regime": REFERENCE_REGIME,
                "reference_order": " > ".join(reference_order) if reference_order else "",
                "signature_order": " > ".join(sig_order) if sig_order else "",
                "ordering_status_vs_reference": compare_orderings(reference_order, sig_order),
                "signature_total_n": int(sig_meta_one.get("total_n", np.nan)) if pd.notna(sig_meta_one.get("total_n", np.nan)) else np.nan,
                "usable_style_count": int(sig_meta_one.get("usable_style_count", np.nan)) if pd.notna(sig_meta_one.get("usable_style_count", np.nan)) else np.nan,
                "community_n": int(sig_meta_one.get("Community", 0)),
                "third_party_n": int(sig_meta_one.get("Third-Party", 0)),
                "gmd_n": int(sig_meta_one.get("GMD", 0)),
                "custom_n": int(sig_meta_one.get("Custom", 0)),
                "study_runner_os_bucket": sig_meta_one.get("study_runner_os_bucket", ""),
                "study_job_count_total_bucket": sig_meta_one.get("study_job_count_total_bucket", ""),
                "study_step_count_exec_bucket": sig_meta_one.get("study_step_count_exec_bucket", ""),
                "interpretive_dominant_grouping_sum": dom_grouping,
            })

selected_signature_summary = pd.DataFrame(selected_rows)
selected_signature_summary.to_csv(OUT_DIR / "step3_selected_signature_summary.csv", index=False)

signature_stability = pd.DataFrame(stability_rows)
signature_stability.to_csv(OUT_DIR / "step3_signature_conclusion_stability.csv", index=False)

# ============================================================
# HUMAN-READABLE NOTES
# ============================================================
notes = []
notes.append("Validation Step 3 — automated conclusion-robustness outputs")
notes.append("")
notes.append(f"Input file: {IN_MAIN}")
notes.append(f"Output folder: {OUT_DIR}")
notes.append("")
notes.append("Files produced:")
for fn in [
    "step3_regime_summary.csv",
    "step3_regime_ordering_vs_base.csv",
    "step3_signature_candidates.csv",
    "step3_selected_signature_summary.csv",
    "step3_signature_conclusion_stability.csv",
]:
    notes.append(f"- {fn}")

notes.append("")
notes.append("Interpretation guide:")
notes.append("1) Regime summaries:")
notes.append("   - Compare medians, IQR, and P95 across all_run_per_style, verdict_complete, controlled_subset, rerun_verdict_complete.")
notes.append("   - The controlled_subset is supported if it is more stable/tighter and later style conclusions are not contradicted.")
notes.append("")
notes.append("2) Regime ordering:")
notes.append("   - preserved / partially_preserved against the controlled_subset supports controller-regime robustness.")
notes.append("")
notes.append("3) Signature candidates:")
notes.append("   - Ranked by total support and usable cross-style overlap.")
notes.append("")
notes.append("4) Within-signature robustness:")
notes.append("   - preserved / partially_preserved against the controlled_subset supports within-shape robustness.")
notes.append("")
notes.append("Cautions:")
notes.append("- Custom may be sparse in some signatures.")
notes.append("- Layer 2 is evaluated only where study_layer2_measurement_mode == measured_step_based.")
notes.append("- Grouping sums are interpretive only.")
notes.append("")
notes.append("Suggested paper use:")
notes.append("- Methodology: mention Step 3 briefly as automated conclusion-robustness checks.")
notes.append("- RQ1: report the actual regime and within-signature evidence from these outputs.")

(OUT_DIR / "step3_notes.txt").write_text("\n".join(notes), encoding="utf-8")

print(f"Done. Outputs saved to: {OUT_DIR}")
print("Selected signatures:", selected_signatures)

Done. Outputs saved to: C:\Android Mobile App\ICST2026_Ext\0.2-Validation\Step 3 - Robustness_Check
Selected signatures: ['5d2a4439b138d7a4', 'e219945f860a08d4']
